# Week 1 Day 2 Challenge - the cheapest model for the agent

The challenge: with a goal of at least 15 of the 25 claims right in every run, using the agent, what is the cheapest model you can find?

This notebook runs the lab's agent, with its instructions, prompt and two tools unchanged, on seven models, three times each, and scores every run against the handlers' decisions the way the lab does. The five "manual" claims need the claims manual, which the agent doesn't have, so 15 of 25 means 15 of the 20 claims it can decide.

## The candidates

The agent makes three model calls for each claim, so a run of 25 claims is 75 calls, and "consistently" needs several runs of each model. Free models on OpenRouter allow 1,000 requests a day once $10 of credit has been bought, so two free models can do three runs each. Gemma 4 31B and 26B (free) were rate-limited upstream and Nemotron 3 Super 120B (free) was refused by its provider when I tried them, so the free pair is Nemotron 3 Ultra 550B and Poolside's Laguna S 2.1. The rest are the cheapest paid models on OpenRouter that support tools, from about $0.02 per million input tokens up to GPT-6 Luna, the lab's choice, with Claude Haiku 5.5 on Anthropic's own API for comparison at the same price as Luna:

| Model | Input, per million tokens | Output, per million tokens |
|---|---|---|
| Nemotron 3 Ultra 550B (free) | $0 | $0 |
| Laguna S 2.1 (free) | $0 | $0 |
| GPT-OSS 20B | $0.018 | $0.09 |
| GLM 5.3 Flash | $0.042 | $0.077 |
| DeepSeek V4.1 Flash, `reasoning_effort="low"` | $0.02 | $1.00 |
| GPT-6 Luna, `reasoning_effort="low"` | $0.10 | $0.50 |
| Claude Haiku 5.5, `reasoning_effort="low"` | $0.10 | $0.50 |

These are OpenRouter's list prices on the day of the run. The cost in the results is what OpenRouter reports for each call, which depends on the provider it routes to; Haiku's is worked out from its tokens.

The free models' providers may train on what they are sent. These claims are made up, so that's fine here, but it rules them out for real claims.

In [1]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.exceptions import ModelRateLimitError
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# This notebook sits two folders below the repo's root
ROOT = Path("../..")

## Connect to OpenRouter and Anthropic

Both keys come from the `.env` file in the `undercover` folder.

In [2]:
load_dotenv(ROOT / ".env", override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

for key in ["OPENROUTER_API_KEY", "ANTHROPIC_API_KEY"]:
    print(f"{key} found" if os.getenv(key) else f"No {key} found. Add it to the .env file")

OPENROUTER_API_KEY found
ANTHROPIC_API_KEY found


## The claims and the policy wording

The same 25 claims as the lab, with the handlers' decisions.

In [3]:
DATA = ROOT / "data"


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("evals/decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


## The agent

The instructions, the prompt and the two tools are the lab's, unchanged.

In [4]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""

In [5]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

## The models

The OpenRouter models go through LangChain's `ChatOpenAI`, as in the lab. Haiku goes through `ChatAnthropic`, with its prices in US dollars per million tokens, since Anthropic doesn't report the cost of a call.

In [6]:
def openrouter(model, **options):
    """A chat model on OpenRouter."""
    return ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5, **options)


MODELS = {
    "Nemotron 3 Ultra 550B (free)": {"llm": openrouter("nvidia/nemotron-3-ultra-550b-a55b:free"), "claims_at_a_time": 2},
    "Laguna S 2.1 (free)": {"llm": openrouter("poolside/laguna-s-2.1:free"), "claims_at_a_time": 2},
    "GPT-OSS 20B": {"llm": openrouter("openai/gpt-oss-20b")},
    "GLM 5.3 Flash": {"llm": openrouter("z-ai/glm-5.3-flash")},
    "DeepSeek V4.1 Flash": {"llm": openrouter("deepseek/deepseek-v4.1-flash", reasoning_effort="low")},
    "GPT-6 Luna": {"llm": openrouter("openai/gpt-6-luna", reasoning_effort="low")},
    "Claude Haiku 5.5": {
        "llm": ChatAnthropic(model="claude-haiku-5-5", reasoning_effort="low", max_tokens=16000, timeout=120, max_retries=5),
        "input_price": 0.10,
        "output_price": 0.50,
    },
}

agents = {
    name: create_agent(model=model["llm"], tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)
    for name, model in MODELS.items()
}

## Scoring

A claim is right when the decision matches the handler's and the payout is within a dollar, as in the lab. A claim with no recorded decision counts as wrong, and so does one where the agent fails or goes round in circles: `recursion_limit` stops it after 20 steps, well past the lab's three model calls. The error is kept, so the results can tell a refusal from a wrong answer. Tokens come from each reply's `usage_metadata`.

Free models on OpenRouter also allow only 20 requests a minute. A claim stopped by that limit isn't the model's fault, so it starts again from the beginning after a minute, and its time is that of the try that finished.

In [7]:
RATE_LIMIT_TRIES = 3


def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def reply_cost(reply, model):
    """What one model call cost: OpenRouter reports it, and Haiku's is worked out from its tokens."""
    reported = reply.response_metadata.get("token_usage", {}).get("cost")
    if reported is not None:
        return reported
    usage = reply.usage_metadata or {}
    return (usage.get("input_tokens", 0) * model["input_price"] + usage.get("output_tokens", 0) * model["output_price"]) / 1_000_000


def decide_with_agent(name, claim, run):
    """Run one model's agent on one claim and check what it recorded against the handler's decision.

    A claim stopped by a rate limit before it recorded anything starts again after a minute, up to three tries.
    """
    for _ in range(RATE_LIMIT_TRIES):
        recorded.pop(claim["claim_id"], None)
        messages = []
        error = ""
        start = time.time()
        try:
            result = agents[name].invoke(
                {"messages": [{"role": "user", "content": make_claim_prompt(claim)}]},
                config={"recursion_limit": 20},
            )
            messages = result["messages"]
        except ModelRateLimitError as exception:
            error = f"{type(exception).__name__}: {exception}"
            if claim["claim_id"] not in recorded:
                time.sleep(60)
                continue
        except Exception as exception:
            error = f"{type(exception).__name__}: {exception}"
        break
    seconds = time.time() - start
    replies = [message for message in messages if message.type == "ai"]
    input_tokens = sum((reply.usage_metadata or {}).get("input_tokens", 0) for reply in replies)
    output_tokens = sum((reply.usage_metadata or {}).get("output_tokens", 0) for reply in replies)
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    return {
        "model": name,
        "run": run,
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "error": error,
        "model_calls": len(replies),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost": sum(reply_cost(reply, MODELS[name]) for reply in replies),
        "seconds": round(seconds, 1),
    }

## Run every model three times

Five claims at a time, each in its own thread, to save waiting on the network, and two at a time on the free models, to keep under their 20 requests a minute; the seconds are still each claim's own. Each finished run is saved to `week1day2_runs.json`, so if the notebook stops, running this cell again carries on from the next run. Delete the file to start again.

In [8]:
N_RUNS = 3
CLAIMS_AT_A_TIME = 5
RUNS_FILE = Path("week1day2_runs.json")

results = json.loads(RUNS_FILE.read_text(encoding="utf-8")) if RUNS_FILE.exists() else []
done = {(result["model"], result["run"]) for result in results}

for name in MODELS:
    for run in range(1, N_RUNS + 1):
        if (name, run) in done:
            continue
        with ThreadPoolExecutor(max_workers=MODELS[name].get("claims_at_a_time", CLAIMS_AT_A_TIME)) as pool:
            run_results = list(pool.map(lambda claim: decide_with_agent(name, claim, run), claims))
        results += run_results
        RUNS_FILE.write_text(json.dumps(results, indent=2), encoding="utf-8")

scored = pd.DataFrame(results)
for (name, run), run_results in scored.groupby(["model", "run"], sort=False):
    right = run_results["right"].sum()
    errors = (run_results["error"] != "").sum()
    print(
        f"{name}, run {run}: {right} of {len(run_results)} right, {errors} errors, "
        f"${run_results['cost'].sum():.4f}, {run_results['seconds'].mean():.0f} seconds a claim"
    )

Nemotron 3 Ultra 550B (free), run 1: 16 of 25 right, 0 errors, $0.0000, 35 seconds a claim
Nemotron 3 Ultra 550B (free), run 2: 17 of 25 right, 0 errors, $0.0000, 31 seconds a claim
Nemotron 3 Ultra 550B (free), run 3: 17 of 25 right, 0 errors, $0.0000, 32 seconds a claim
Laguna S 2.1 (free), run 1: 15 of 25 right, 0 errors, $0.0000, 82 seconds a claim
Laguna S 2.1 (free), run 2: 14 of 25 right, 1 errors, $0.0000, 76 seconds a claim
Laguna S 2.1 (free), run 3: 13 of 25 right, 0 errors, $0.0000, 55 seconds a claim
GPT-OSS 20B, run 1: 12 of 25 right, 0 errors, $0.0094, 59 seconds a claim
GPT-OSS 20B, run 2: 13 of 25 right, 0 errors, $0.0055, 21 seconds a claim
GPT-OSS 20B, run 3: 11 of 25 right, 0 errors, $0.0049, 22 seconds a claim
GLM 5.3 Flash, run 1: 15 of 25 right, 0 errors, $0.0159, 18 seconds a claim
GLM 5.3 Flash, run 2: 17 of 25 right, 0 errors, $0.0121, 16 seconds a claim
GLM 5.3 Flash, run 3: 19 of 25 right, 0 errors, $0.0136, 17 seconds a claim
DeepSeek V4.1 Flash, run 1: 18 

## The results

Each model's fewest, mean and most claims right across its runs, with the cost and time for one claim. A model passes when its fewest is at least 15.

In [9]:
runs = scored.groupby(["model", "run"], sort=False).agg(right=("right", "sum")).reset_index()

summary = runs.groupby("model", sort=False)["right"].agg(fewest="min", mean="mean", most="max")
per_claim = scored.groupby("model", sort=False).agg(
    cost_per_claim=("cost", "mean"),
    seconds_per_claim=("seconds", "mean"),
    model_calls_per_claim=("model_calls", "mean"),
    nothing_recorded=("decision", lambda decisions: (decisions == "nothing recorded").sum()),
    errors=("error", lambda errors: (errors != "").sum()),
)
summary = summary.join(per_claim)
summary["passes"] = summary["fewest"] >= 15
summary["mean"] = summary["mean"].round(1)
summary["cost_per_claim"] = summary["cost_per_claim"].map(lambda cost: f"${cost:.5f}")
summary["seconds_per_claim"] = summary["seconds_per_claim"].round(1)
summary["model_calls_per_claim"] = summary["model_calls_per_claim"].round(1)
summary

,fewest,mean,most,cost_per_claim,seconds_per_claim,model_calls_per_claim,nothing_recorded,errors,passes
model,,,,,,,,,
Nemotron 3 Ultra 550B (free),16,16.7,17,$0.00000,32.7,3.0,1,0,True
Laguna S 2.1 (free),13,14.0,15,$0.00000,71.2,3.1,3,1,False
GPT-OSS 20B,11,12.0,13,$0.00026,34.4,3.1,0,0,False
GLM 5.3 Flash,15,17.0,19,$0.00056,16.9,3.0,0,0,True
DeepSeek V4.1 Flash,18,18.7,19,$0.00106,7.0,3.0,0,0,True
GPT-6 Luna,17,17.7,19,$0.00030,6.9,3.0,0,0,True
Claude Haiku 5.5,16,17.0,18,$0.00203,7.5,3.0,0,0,True


In [10]:
fig = px.bar(
    runs.astype({"run": str}),
    x="model",
    y="right",
    color="run",
    barmode="group",
    title="Claims right in each run, out of 25",
)
fig.add_hline(y=15, line_dash="dash", annotation_text="the goal: 15")
fig.show()

### Right by difficulty

The mean number right at each level of difficulty, across the runs. There are 15 routine claims, 5 hard and 5 manual.

In [11]:
by_difficulty = scored.pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum", sort=False)
(by_difficulty / N_RUNS).round(1).reindex(columns=["routine", "hard", "manual"])

difficulty,routine,hard,manual
model,,,
Nemotron 3 Ultra 550B (free),15.0,1.3,0.3
Laguna S 2.1 (free),12.0,1.7,0.3
GPT-OSS 20B,12.0,0.0,0.0
GLM 5.3 Flash,14.7,1.7,0.7
DeepSeek V4.1 Flash,15.0,3.7,0.0
GPT-6 Luna,13.7,4.0,0.0
Claude Haiku 5.5,13.7,3.0,0.3


## The claims each model got wrong

For each claim, the number of runs in which each model got it wrong, out of three, with the handler's decision. Claims every model got right in every run are left out.

In [12]:
wrong = scored.assign(wrong=~scored["right"])
misses = wrong.pivot_table(index="claim_id", columns="model", values="wrong", aggfunc="sum", sort=False)
misses = misses[misses.sum(axis=1) > 0]
handlers = pd.DataFrame(
    {
        claim_id: {
            "difficulty": labels[claim_id]["difficulty"],
            "handler": f"{labels[claim_id]['decision']} ${labels[claim_id]['payout']:,.2f}",
        }
        for claim_id in misses.index
    }
).T
handlers.join(misses)

,difficulty,handler,Nemotron 3 Ultra 550B (free),Laguna S 2.1 (free),GPT-OSS 20B,GLM 5.3 Flash,DeepSeek V4.1 Flash,GPT-6 Luna,Claude Haiku 5.5
CLM-1001,routine,approve $420.00,0,1,0,0,0,0,0
CLM-1002,routine,approve $580.00,0,2,3,1,0,0,0
CLM-1004,routine,approve $186.00,0,0,0,0,0,3,3
CLM-1006,routine,approve $425.00,0,1,0,0,0,0,0
CLM-1008,routine,"approve $2,240.00",0,0,1,0,0,0,0
CLM-1009,routine,approve $930.00,0,1,1,0,0,1,0
CLM-1010,routine,decline $0.00,0,1,3,0,0,0,0
CLM-1011,hard,"approve $1,150.00",3,1,3,2,0,1,1
CLM-1012,routine,decline $0.00,0,2,1,0,0,0,0
CLM-1014,hard,approve $605.00,0,0,3,0,0,0,0


To read a model's rationale on its misses, pass its name and a run to `show_misses`.

In [13]:
def show_misses(name, run):
    """Show each claim a model got wrong in one run, with its rationale and the handler's reason."""
    for result in scored[(scored["model"] == name) & (scored["run"] == run) & ~scored["right"]].to_dict("records"):
        label = labels[result["claim_id"]]
        display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses("Nemotron 3 Ultra 550B (free)", 2)

**CLM-1011, hard**: approve $1,110.00, where the handler said approve $1,150.00

- Model: The claim is for lost checked baggage during the covered trip (June 2-10, 2026). The Premium plan provides $1,500 maximum for Baggage and Personal Effects with a $300 per-item limit and a $50 deductible. Items that form a set (suit jacket+trousers, camera body+lens, four shirts) count as one item each per Section 5. Eligible amounts per item are capped at $300: suit $300, camera set $300, shirts $300, shoes $230, suitcase $260, cufflinks $120 = $1,510 total. Carrier Meridian Air paid $350, which is deducted. Then the $50 deductible applies. Result: $1,510 - $350 - $50 = $1,110, which is under the $1,500 maximum. All required documents (carrier report, settlement letter, receipts) are provided. Claim filed within 90 days.
- Handler: The carrier has declared the bag lost and its report is on file. A suit counts as one item, and the receipt shows the jacket and trousers were sold as a two piece tuxedo, so together they count at the $300 per-item limit on Premium. A camera with its lens is one item too, so the body and the lens together count at $300. The four shirts are four items at $85 each, all under the limit, so that line is paid in full. The carrier's $350 comes off, then the $50 deductible. The payout is over $1,000, so payment needs sign-off.
- Tests: Payout arithmetic: the per-item limit applies to items, and the claim's lines are not items. Two sets are split across lines and count as one item each, and one line holds four separate items


**CLM-1018, manual**: approve $330.00, where the handler said refer $0.00

- Model: The claim is for theft of a leather jacket ($420) and canvas daypack ($80) during the covered trip (June 22, 2026). The customer has a Premium plan with $1,500 maximum for Baggage and Personal Effects, $300 per-item limit, and a $50 deductible. The theft was reported to Carabinieri within 24 hours and a police report is attached (section 5). The per-item limit caps the jacket at $300; the daypack is $80. Eligible amount: $300 + $80 = $380. Minus $50 deductible = $330. This is within the plan maximum.
- Handler: A theft from the person, reported to the police within the hour, with the report on file. The manual asks for proof of value for any item claimed above $100, which is a purchase receipt or a card statement, and the jacket at $420 has none. Document request: proof of what the jacket cost, for example a cash withdrawal record from that morning together with the stallholder's details. The daypack at $80 needs no proof. The claim appears payable at $330 once the proof arrives: the jacket at the $300 limit plus $80, less the $50 deductible.
- Tests: A rule found only in the claims manual: proof of value for items claimed above $100


**CLM-1019, manual**: approve $210.00, where the handler said refer $0.00

- Model: The claim is for a 9 hour 20 minute delay on the outbound flight from Miami to Bogota, which exceeds the Premium plan's 6-hour waiting period for Trip Delay. The cause (crew duty time limit requiring replacement crew) is a covered reason under section 3. The expenses ($112 day room, $58 lunch, $40 taxi) are reasonable extra costs for meals, accommodation, and local transport during the delay, totaling $210, well within the $800 maximum. The carrier's delay confirmation and receipts are provided. No deductible applies to Trip Delay per section 1.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The claim is for prescription sunglasses stolen by a monkey at Uluwatu Temple during the covered trip. The First plan provides Baggage and Personal Effects coverage up to $3,000 with a $500 per-item limit and $0 deductible. The replacement cost of $240 is within the per-item limit. However, section 5 requires theft to be reported to local police within 24 hours and a copy of the police report to be provided. The customer states the police refused to take a report about monkeys and instead provided a guide's letter. The policy wording does not address whether a guide's letter suffices when police decline to file a report. This ambiguity requires a colleague's judgment on whether the documentation requirement is satisfied.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1022, manual**: refer $0.00, where the handler said approve $155.00

- Model: The carrier report confirms a 30-hour delay due to severe weather (flash flooding), which exceeds the 3-hour waiting period for the First plan and is a covered cause under section 3. The rebooking fee receipt ($85) is provided. However, receipts for the $140 in meal expenses are missing, as required by section 3. Per section 11, the claim must wait for the missing documents.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1023, hard**: approve $61.00, where the handler said refer $0.00

- Model: The customer holds a First plan with a 3-hour waiting period and $1,500 maximum for Trip Delay. The flight was delayed 9 hours due to a mechanical fault (fuel pump), which is a covered cause under section 3. Meal receipts and the carrier's delay certificate are provided. The claimed $61 for meals during the delay is within the plan maximum and no deductible applies to Trip Delay. All requirements are satisfied.
- Handler: The delay of 9 hours 17 minutes is well past the 3 hour waiting period on First and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the fuel pump fault rests on the customer's account alone. Document request: Northwind's confirmation of the reason for the delay. The claim appears payable at $61 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


**CLM-1024, manual**: approve $250.00, where the handler said refer $0.00

- Model: The theft of the smartphone occurred during the covered trip (July 8, 2026) in Bogota, Colombia. The police report was filed within 24 hours as required (Section 5), and the phone was snatched from the claimant's hand, so it was not left unattended. The Premium plan has a $1,500 maximum for Baggage and Personal Effects with a $300 per-item limit (no Electronics Plus add-on). The eligible amount is the per-item limit of $300. After applying the $50 deductible (Section 1), the payout is $250.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


**CLM-1025, hard**: approve $71.00, where the handler said refer $0.00

- Model: The Economy plan has a 12-hour waiting period for Trip Delay; the flight was delayed 14 hours 15 minutes due to a crew problem (first officer duty hours), which is a covered reason under section 3. The carrier's delay certificate confirms the delay and cause. Meal receipts for $71 are provided and are reasonable extra costs during the delay. The delay occurred on the return journey past the policy return date, but section 3 states coverage continues until the traveler gets home. The claim is within the plan's $300 maximum and deductible does not apply to Trip Delay. All required documents are attached.
- Handler: The delay of 14 hours 15 minutes is past the 12 hour waiting period on Economy and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the crew problem rests on the customer's account alone. Document request: Skylark's confirmation of the reason for the delay. The claim appears payable at $71 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


## What I found

**The cheapest model that gets at least 15 right in every run is Nemotron 3 Ultra 550B (free), at $0.** It got 16, 17 and 17 of 25, with all 15 routine claims right in every run. It suits a lab, not the product: it took about 33 seconds a claim, the free tier allows 20 requests a minute, and its provider may train on what it is sent.

**The cheapest paid model that passes is GPT-6 Luna, the lab's own choice**, at $0.0003 a claim, under a cent for the 25 claims, and 7 seconds a claim. Ranked by what each claim actually cost:

| Model | Right in each run | Cost a claim | Passes |
|---|---|---|---|
| Nemotron 3 Ultra 550B (free) | 16, 17, 17 | $0 | Yes |
| Laguna S 2.1 (free) | 15, 14, 13 | $0 | No |
| GPT-OSS 20B | 12, 13, 11 | $0.00026 | No |
| GPT-6 Luna | 17, 19, 17 | $0.00030 | Yes |
| GLM 5.3 Flash | 15, 17, 19 | $0.00056 | Just |
| DeepSeek V4.1 Flash | 18, 19, 19 | $0.00106 | Yes, and the most accurate |
| Claude Haiku 5.5 | 16, 17, 18 | $0.00203 | Yes |

What surprised me:

- **The list price didn't predict the cost.** GPT-OSS 20B and GLM 5.3 Flash list at a fraction of Luna's input price, but they think at length, about 1,700 to 1,800 output tokens a claim against Luna's 330, and OpenRouter billed Luna below its list price. That is most likely the provider caching the system prompt, which is the same on every call; I didn't record cached tokens, so that part is an inference. GLM ended up costing almost twice as much as Luna, and GPT-OSS saved four thousandths of a cent a claim and failed.
- **Haiku 5.5 has the same list price as Luna and cost about seven times as much a claim.** Anthropic counted about 14,500 input tokens a claim for the same prompt, against about 8,400 on Luna, and without cache control set up in `ChatAnthropic`, every call pays full price for the policy wording.
- **The hard claims separate the models more than the routine ones.** On the 15 routine claims the models are close, from 12 to 15 right. CLM-1023 and CLM-1025 are claims the handlers referred, and most models approved them: Nemotron, Laguna, GPT-OSS, GLM and Haiku got CLM-1025 wrong in every run, Luna got both right in every run, and DeepSeek missed each once. DeepSeek is right on 3.7 of the 5 hard claims on average, Luna on 4.0.
- **No model gets the manual claims**, as the lab said: they need the claims manual.
- **Three runs is a small sample.** GLM went 15, 17, 19, so a single miss more in its first run would have failed it. I'd want more runs before choosing between models a point or two apart.

For the product I'd still pick DeepSeek V4.1 Flash, as the brief does: it had the highest worst run (18), at about a tenth of a cent a claim. If the cost per claim mattered more than a point of accuracy, Luna is the one.

The runs cost $0.16 on OpenRouter and $0.15 on Anthropic. Nemotron's first run ran five claims at a time, before I found the free models' limit of 20 requests a minute; it had no errors, so I kept it, and threw away a second run that the limit had spoiled.